In [8]:
from torch.utils.data import Dataset

max_dataset_size = 200000

class LCSTS(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                if idx >= max_dataset_size:
                    break
                items = line.strip().split('!=!')
                assert len(items) == 2
                Data[idx] = {
                    'title': items[0],
                    'content': items[1]
                }
        return Data
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_data = LCSTS('data/lcsts_tsv/data1.tsv')
valid_data = LCSTS('data/lcsts_tsv/data2.tsv')
test_data = LCSTS('data/lcsts_tsv/data3.tsv')

In [9]:
print(f'train set size: {len(train_data)}')
print(f'valid set size: {len(valid_data)}')
print(f'test set size: {len(test_data)}')
print(next(iter(train_data)))

train set size: 200000
valid set size: 10666
test set size: 1106
{'title': '修改后的立法法全文公布', 'content': '新华社受权于18日全文播发修改后的《中华人民共和国立法法》，修改后的立法法分为“总则”“法律”“行政法规”“地方性法规、自治条例和单行条例、规章”“适用与备案审查”“附则”等6章，共计105条。'}


In [5]:
from transformers import AutoTokenizer

model_checkpoint = "./model/csebuetnlp/mT5_multilingual_XLSum"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/opt/homebrew/Caskroom/miniconda/base/envs/transformers_learn/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
The tokenizer you are loading from './model/csebuetnlp/mT5_multilingual_XLSum' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [6]:
inputs = tokenizer("我叫张三，在苏州大学学习计算机。")
print(inputs)
print(tokenizer.convert_ids_to_tokens(inputs.input_ids))

{'input_ids': [259, 3003, 27333, 8922, 2092, 261, 1083, 117707, 9792, 24920, 123553, 306, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['▁', '我', '叫', '张', '三', ',', '在', '苏州', '大学', '学习', '计算机', '。', '</s>']


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM

max_input_length = 512
max_target_length = 64

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using {device} device')

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model = model.to(device)

# 由于本文直接使用 Transformers 库自带的 AutoModelForSeq2SeqLM 函数来构建模型，因此我们将每一个 batch 中的数据都处理为该模型可接受的格式：一个包含 'attention_mask'（编码器掩码）、'input_ids'（编码器输入）、'labels'（期望的输出） 和 'decoder_input_ids'（解码器输入） 键的字典。 
def collate_fn(batch_samples):
    batch_inputs, batch_targets = [], []
    for sample in batch_samples:
        batch_inputs.append(sample['content'])
        batch_targets.append(sample['title'])
    batch_data = tokenizer(
        batch_inputs, 
        padding=True,  # 在T5和mT5模型中，编码器（Encoder）的 input_ids 填充位（Padding Token ID）默认是 0。tokenizer.pad_token_id为0
        max_length=max_input_length,
        truncation=True, 
        return_tensors="pt"
    )
    with tokenizer.as_target_tokenizer(): # 上下文管理器，告诉分词器（tokenizer）接下来的分词操作是针对目标语言的（原理：对于像 mT5 这样的多语言模型，源文本和目标文本可能属于不同语言。有些分词器在处理目标文本时会使用不同的特殊 token 或处理逻辑。虽然在某些模型中它和普通分词效果一样，但在 Seq2Seq 任务中这是标准且安全的写法。）
        labels = tokenizer(
            batch_targets, 
            padding=True, 
            max_length=max_target_length,
            truncation=True, 
            return_tensors="pt"
        )["input_ids"]  # 将标题（batch_targets）转换为数字 ID 序列。（batch_size, seq_len）
        # 根据labels自动生成解码器的输入（进行右移）
        # 大部分情况，即使没有decoder_input_ids，模型也能正常训练，它会自动做下面这一步
        batch_data['decoder_input_ids'] = model.prepare_decoder_input_ids_from_labels(labels)
        """
        labels == tokenizer.eos_token_id：生成一个与 labels 形状相同的布尔张量（True/False），其中 </s> 的位置为 True。
        torch.where(...)：返回一个元组，包含所有 True 元素的坐标。返回值的第 0 维：行索引（即这个 </s> 属于 batch 中的第几个样本）。返回值的第 1 维：列索引（即这个 </s> 在该句子中的具体位置）。
        [1]：我们只关心列索引（即每个句子的结束位置），所以取索引为 1 的部分。
        假设结果： end_token_index 可能是 tensor([12, 45, 30, 22])，表示第 1 句在索引 12 结束，第 2 句在 45 结束，依此类推。
        """
        end_token_index = torch.where(labels == tokenizer.eos_token_id)[1]
        for idx, end_idx in enumerate(end_token_index):
            # 将EOS之后的所有填充部分赋值为-100（在 PyTorch 的交叉熵损失函数（CrossEntropyLoss）中，ignore_index 的默认值通常是 -100，这是工程上的约定俗成）
            labels[idx][end_idx+1:] = -100
        batch_data['labels'] = labels
    return batch_data

train_dataloader = DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=collate_fn)
valid_dataloader = DataLoader(valid_data, batch_size=4, shuffle=False, collate_fn=collate_fn)

Using mps device


In [11]:
from tqdm.auto import tqdm

def train_loop(dataloader, model, optimizer, lr_scheduler, epoch, total_loss):
    progress_bar = tqdm(range(len(dataloader)))
    progress_bar.set_description(f'loss: {0:>7f}')
    finish_batch_num = (epoch-1) * len(dataloader)
    
    model.train()
    for batch, batch_data in enumerate(dataloader, start=1):
        batch_data = batch_data.to(device)
        outputs = model(**batch_data)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_description(f'loss: {total_loss/(finish_batch_num + batch):>7f}')
        progress_bar.update(1)
    return total_loss

In [12]:
from rouge import Rouge

generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

rouge = Rouge()

scores = rouge.get_scores(
    hyps=[generated_summary], refs=[reference_summary]
)[0]
print(scores)

{'rouge-1': {'r': 1.0, 'p': 0.8571428571428571, 'f': 0.9230769181065088}, 'rouge-2': {'r': 0.8, 'p': 0.6666666666666666, 'f': 0.7272727223140496}, 'rouge-l': {'r': 1.0, 'p': 0.8571428571428571, 'f': 0.9230769181065088}}


In [13]:
from rouge import Rouge

generated_summary = "我在苏州大学学习计算机，苏州大学很美丽。"
reference_summary = "我在环境优美的苏州大学学习计算机。"

rouge = Rouge()

# rouge 库默认使用空格进行分词，因此无法处理中文、日文等语言，最简单的办法是按字进行切分，当然也可以使用分词器分词后再进行计算，否则会计算出不正确的ROUGE值
TOKENIZE_CHINESE = lambda x: ' '.join(x)  # 把一句话用空格连接起来（每两个字中间都加一个空格）

# from transformers import AutoTokenizer
# model_checkpoint = "csebuetnlp/mT5_multilingual_XLSum"
# tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# TOKENIZE_CHINESE = lambda x: ' '.join(
#     tokenizer.convert_ids_to_tokens(tokenizer(x).input_ids, skip_special_tokens=True)
# )

scores = rouge.get_scores(
    hyps=[TOKENIZE_CHINESE(generated_summary)], 
    refs=[TOKENIZE_CHINESE(reference_summary)]
)[0]
print('ROUGE:', scores)
scores = rouge.get_scores(
    hyps=[generated_summary], 
    refs=[reference_summary]
)[0]
print('wrong ROUGE:', scores)

ROUGE: {'rouge-1': {'r': 0.75, 'p': 0.8, 'f': 0.7741935433922998}, 'rouge-2': {'r': 0.5625, 'p': 0.5625, 'f': 0.562499995}, 'rouge-l': {'r': 0.6875, 'p': 0.7333333333333333, 'f': 0.7096774143600416}}
wrong ROUGE: {'rouge-1': {'r': 0.0, 'p': 0.0, 'f': 0.0}, 'rouge-2': {'r': 0.0, 'p': 0.0, 'f': 0.0}, 'rouge-l': {'r': 0.0, 'p': 0.0, 'f': 0.0}}


In [17]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model = model.to(device)

article_text = """
受众在哪里，媒体就应该在哪里，媒体的体制、内容、技术就应该向哪里转变。
媒体融合关键是以人为本，即满足大众的信息需求，为受众提供更优质的服务。
这就要求媒体在融合发展的过程中，既注重技术创新，又注重用户体验。
"""
# mT5 模型推理流程：代码的执行逻辑分为 预处理、模型生成、后处理 三个阶段

# 预处理：tokenizer 将你的中文文本切分成一个个子词（Subwords），并转换成词表中的数字 ID。
input_ids = tokenizer(
    article_text,
    return_tensors="pt",  # 确保输出的是 GPU 能够处理的张量格式。
    truncation=True,
    max_length=512
)

# 模型生成：
# Encoder 端： 模型先将 input_ids 全部读入，通过多层自注意力机制，把这篇关于“媒体融合”的文章转化成一堆高维度的隐藏向量。
# Decoder 端：从 <s> 开始，根据 Encoder 的信息，在 25 万个词里找最可能的 4 个词（Beam 1-4）。循环往复，直到生成了 </s> 结束符，或者达到了 max_length=32 的硬限制。
generated_tokens = model.generate(
    input_ids["input_ids"],
    attention_mask=input_ids["attention_mask"],
    max_length=32,    # 限制生成的摘要长度
    no_repeat_ngram_size=2,  # 惩罚 2-gram 重复，让话术更自然，确保不会出现“媒体媒体”或“融合融合”这种尴尬的词组重复。
    num_beams=4  # 启动 4 条候选路径并行搜索
)

# 后处理阶段：decode还原文字
summary = tokenizer.decode(
    generated_tokens[0],  # 取出4条路径中最优的一条（模型默认会按照得分从高到低排列结果）
    skip_special_tokens=True,  # 自动过滤掉 <s>, </s>, <pad>（如果不加这个，你的摘要开头和结尾可能会出现 </s> 字符，加上它能得到干净的纯文本。）
    clean_up_tokenization_spaces=False # 保持原始空格处理方式（对中文影响较小）
)
print(summary)

/opt/homebrew/Caskroom/miniconda/base/envs/transformers_learn/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
The tokenizer you are loading from './model/csebuetnlp/mT5_multilingual_XLSum' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


媒体融合发展是当下中国面临的一大难题。


In [ ]:
article_texts = [
"""
受众在哪里，媒体就应该在哪里，媒体的体制、内容、技术就应该向哪里转变。
媒体融合关键是以人为本，即满足大众的信息需求，为受众提供更优质的服务。
这就要求媒体在融合发展的过程中，既注重技术创新，又注重用户体验。
""",
"""
新华社受权于18日全文播发修改后的《中华人民共和国立法法》，
修改后的立法法分为“总则”“法律”“行政法规”“地方性法规、
自治条例和单行条例、规章”“适用与备案审查”“附则”等6章，共计105条。
"""
]

input_ids = tokenizer(
    article_texts,
    padding=True, 
    return_tensors="pt",
    truncation=True,
    max_length=512
)
generated_tokens = model.generate(
    input_ids["input_ids"],
    attention_mask=input_ids["attention_mask"],
    max_length=32,
    no_repeat_ngram_size=2,
    num_beams=4
)
# batch_decode, num_return_sequences默认是1，也就是返回每个样本4条候选路径的最优路径，返回的shape为：（batch_size, seq_len）；也可以设置num_return_sequences为4，返回的shape:(batch_size*4, seq_len)，每个样本的4条路径按分值降序排
summarys = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)
print(summarys)

['媒体融合发展是当下中国面临的一大难题。', '中国官方新华社周一(18日)全文播发修改后的《中华人民共和国立法法》。']


In [19]:
import numpy as np
from rouge import Rouge

rouge = Rouge()

def test_loop(dataloader, model):
    preds, labels = [], []
    
    model.eval()  # 将模型切换到评估模式，这会关闭dropout和batch normalization等训练专用操作
    for batch_data in tqdm(dataloader): # 遍历测试数据加载器，tqdm会在控制台显示进度条
        batch_data = batch_data.to(device)
        with torch.no_grad():  # 开启上下文管理器，禁用梯度计算，这能显著减少显存占用并加快推理速度
            generated_tokens = model.generate(
                batch_data["input_ids"],
                attention_mask=batch_data["attention_mask"],
                max_length=max_target_length,
                num_beams=4,
                no_repeat_ngram_size=2,
            ).cpu().numpy()  # 生成数据后立即移回CPU并转为numpy数组，方便后续处理
        # 某些模型版本可能返回元组，如果是，则只取第一部分（生成的 IDs）
        if isinstance(generated_tokens, tuple): 
            generated_tokens = generated_tokens[0] 
        label_tokens = batch_data["labels"].cpu().numpy() # 获取该批次的真实标签

        # 将生成的数字ID还原成中文文本
        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        # 在pytorch的损失计算中，-100常用来代表“忽略此位置”；但分词器无法处理-100，因此将其替换为pad_token_id
        label_tokens = np.where(label_tokens != -100, label_tokens, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(label_tokens, skip_special_tokens=True)

        # # 中文摘要评估的“常规操作”：在字与字之间插入空格
        preds += [' '.join(pred.strip()) for pred in decoded_preds]
        labels += [' '.join(label.strip()) for label in decoded_labels]

    # 计算所有样本的ROUGE-1, ROUGE-2, ROUGE-L平均分   
    # 如果没有这个参数，get_scores会返回一个列表，包含每一个样本的得分。设置True后，它会自动帮你算好整个数据集的平均水平
    scores = rouge.get_scores(hyps=preds, refs=labels, avg=True)
    # 只取F1值
    result = {key: value['f'] * 100 for key, value in scores.items()}
    # 计算 ROUGE-1/2/L 三者的平均值，作为一个综合评分指标
    result['avg'] = np.mean(list(result.values()))
    print(f"Rouge1: {result['rouge-1']:>0.2f} Rouge2: {result['rouge-2']:>0.2f} RougeL: {result['rouge-l']:>0.2f}\n")
    return result

In [22]:
test_data = LCSTS('./data/lcsts_tsv/data3.tsv')
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=False, collate_fn=collate_fn)

test_loop(test_dataloader, model)

  0%|          | 0/35 [00:00<?, ?it/s]

/opt/homebrew/Caskroom/miniconda/base/envs/transformers_learn/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:4174: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Rouge1: 23.85 Rouge2: 12.27 RougeL: 20.91



{'rouge-1': 23.845414422286478,
 'rouge-2': 12.26839939126819,
 'rouge-l': 20.908546037034224,
 'avg': np.float64(19.00745328352963)}

In [ ]:
from transformers import get_scheduler
from torch.optim import AdamW

learning_rate = 2e-5
epoch_num = 3

optimizer = AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=epoch_num*len(train_dataloader),
)

total_loss = 0.
best_avg_rouge = 0.
for t in range(epoch_num):
    print(f"Epoch {t+1}/{epoch_num}\n-------------------------------")
    total_loss = train_loop(train_dataloader, model, optimizer, lr_scheduler, t+1, total_loss)
    valid_rouge = test_loop(valid_dataloader, model)
    print(valid_rouge)
    rouge_avg = valid_rouge['avg']
    if rouge_avg > best_avg_rouge:
        best_avg_rouge = rouge_avg
        print('saving new weights...\n')
        torch.save(model.state_dict(), f'epoch_{t+1}_valid_rouge_{rouge_avg:0.4f}_model_weights.bin')
print("Done!")

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import get_scheduler
from torch.optim import AdamW
from tqdm.auto import tqdm
from rouge import Rouge
import random
import numpy as np
import os

from transformers import Adafactor

max_dataset_size = 10000

max_input_length = 512
# max_input_length = 128

max_target_length = 32
# max_target_length = 16

# train_batch_size = 8
train_batch_size = 1

test_batch_size = 8
learning_rate = 2e-5
epoch_num = 3

# beam_size = 4
beam_size = 2

no_repeat_ngram_size = 2

seed = 5
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using {device} device')

class LCSTS(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                if idx >= max_dataset_size:
                    break
                items = line.strip().split('!=!')
                assert len(items) == 2
                Data[idx] = {
                    'title': items[0],
                    'content': items[1]
                }
        return Data
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_data = LCSTS('./data/lcsts_tsv/data1.tsv')
valid_data = LCSTS('./data/lcsts_tsv/data2.tsv')
test_data = LCSTS('./data/lcsts_tsv/data3.tsv')

model_checkpoint = "./model/csebuetnlp/mT5_multilingual_XLSum"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model = model.to(device)

model.config.use_cache = False # 训练时禁用cache

def collote_fn(batch_samples):
    batch_inputs, batch_targets = [], []
    for sample in batch_samples:
        batch_inputs.append(sample['content'])
        batch_targets.append(sample['title'])
    batch_data = tokenizer(
        batch_inputs, 
        padding=True, 
        max_length=max_input_length,
        truncation=True, 
        return_tensors="pt"
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch_targets, 
            padding=True, 
            max_length=max_target_length,
            truncation=True, 
            return_tensors="pt"
        )["input_ids"]
        # batch_data['decoder_input_ids'] = model.prepare_decoder_input_ids_from_labels(labels)
        end_token_index = torch.where(labels == tokenizer.eos_token_id)[1]
        for idx, end_idx in enumerate(end_token_index):
            labels[idx][end_idx+1:] = -100
        batch_data['labels'] = labels
    return batch_data

train_dataloader = DataLoader(train_data, batch_size=train_batch_size, shuffle=True, collate_fn=collote_fn)
valid_dataloader = DataLoader(valid_data, batch_size=test_batch_size, shuffle=False, collate_fn=collote_fn)

def train_loop(dataloader, model, optimizer, 
# lr_scheduler, 
epoch, total_loss):
    progress_bar = tqdm(range(len(dataloader)))
    progress_bar.set_description(f'loss: {0:>7f}')
    finish_batch_num = (epoch-1) * len(dataloader)
    
    model.train()
    for batch, batch_data in enumerate(dataloader, start=1):
        batch_data = batch_data.to(device)
        with torch.autocast(device_type="mps", dtype=torch.float16):
            outputs = model(**batch_data)
            loss = outputs.loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        
        optimizer.step()
        # lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_description(f'loss: {total_loss/(finish_batch_num + batch):>7f}')
        progress_bar.update(1)
    return total_loss

rouge = Rouge()

def test_loop(dataloader, model, mode='Test'):
    assert mode in ['Valid', 'Test']
    preds, labels = [], []
    
    model.eval()
    for batch_data in tqdm(dataloader):
        batch_data = batch_data.to(device)
        with torch.no_grad():
            generated_tokens = model.generate(
                batch_data["input_ids"],
                attention_mask=batch_data["attention_mask"],
                max_length=max_target_length,
                num_beams=beam_size,
                no_repeat_ngram_size=no_repeat_ngram_size,
            ).cpu().numpy()
        if isinstance(generated_tokens, tuple):
            generated_tokens = generated_tokens[0]
        label_tokens = batch_data["labels"].cpu().numpy()

        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        label_tokens = np.where(label_tokens != -100, label_tokens, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(label_tokens, skip_special_tokens=True)

        preds += [' '.join(pred.strip()) for pred in decoded_preds]
        labels += [' '.join(label.strip()) for label in decoded_labels]
    scores = rouge.get_scores(hyps=preds, refs=labels, avg=True)
    result = {key: value['f'] * 100 for key, value in scores.items()}
    result['avg'] = np.mean(list(result.values()))
    print(f"{mode} Rouge1: {result['rouge-1']:>0.2f} Rouge2: {result['rouge-2']:>0.2f} RougeL: {result['rouge-l']:>0.2f}\n")
    return result

# optimizer = AdamW(model.parameters(), lr=learning_rate)
# lr_scheduler = get_scheduler(
#     "linear",
#     optimizer=optimizer,
#     num_warmup_steps=0,
#     num_training_steps=epoch_num*len(train_dataloader),
# )

# 冻结参数这个效果最好，显存不会爆掉了，就是训练时间特别长
for p in model.encoder.parameters():
    p.requires_grad = False 

optimizer = Adafactor(
    model.parameters(),
    scale_parameter=True,
    relative_step=True,
    warmup_init=True,
    lr=None,
)
# lr_scheduler = AdafactorSchedule(optimizer)

total_loss = 0.
best_avg_rouge = 0.
for t in range(epoch_num):
    print(f"Epoch {t+1}/{epoch_num}\n-------------------------------")
    total_loss = train_loop(train_dataloader, model, optimizer, t+1, total_loss)
    valid_rouge = test_loop(valid_dataloader, model, mode='Valid')
    rouge_avg = valid_rouge['avg']
    if rouge_avg > best_avg_rouge:
        best_avg_rouge = rouge_avg
        print('saving new weights...\n')
        torch.save(model.state_dict(), f'epoch_{t+1}_valid_rouge_{rouge_avg:0.4f}_model_weights.bin')
print("Done!")

Using mps device


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/opt/homebrew/Caskroom/miniconda/base/envs/transformers_learn/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warn

Epoch 1/3
-------------------------------


  0%|          | 0/10000 [00:00<?, ?it/s]

/opt/homebrew/Caskroom/miniconda/base/envs/transformers_learn/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:4174: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


KeyboardInterrupt: 